# EventGen - full pipeline on Colab / Kaggle (GPU)
Runtime ▸ Change runtime type ▸ **T4 GPU**. Upload `EventGen.zip` (the project folder) to the session
or to Google Drive, then run the cells in order. Every step is a script of `scripts/`, so the
notebook and the command line give identical results.

In [ ]:
# Option A: zip uploaded to the Colab session
!unzip -q -o /content/EventGen.zip -d /content/ && echo unzipped
# Option B: from Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# !unzip -q -o /content/drive/MyDrive/EventGen.zip -d /content/
%cd /content/EventGen
!pip -q install -r requirements.txt
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. EDA of every source and 2. cleaning → pairs → split → vocabularies

In [ ]:
!python scripts/01_eda.py | tail -5
!python scripts/02_prepare_data.py

## 3. Training (E1 baseline, E2 attention)
Same data, same embedding/hidden sizes, same optimiser, same teacher-forcing ratio and the same
early-stopping rule for both models. On a T4 one epoch takes about 1-2 min; `--max_minutes` is a safety net.

In [ ]:
!python scripts/03_train.py --model baseline --epochs 30 --max_minutes 60

In [ ]:
!python scripts/03_train.py --model attention --epochs 30 --max_minutes 80

### Optional - teacher-forcing study (0.0 / 1.0, attention model)
Results land in `models/attention_tf0.0` and `models/attention_tf1.0` and are evaluated automatically.

In [ ]:
RUN_TF_STUDY = False
if RUN_TF_STUDY:
    !python scripts/03_train.py --model attention --tf 0.0 --tag tf0.0 --epochs 15 --max_minutes 40
    !python scripts/03_train.py --model attention --tf 1.0 --tag tf1.0 --epochs 15 --max_minutes 40

## 4. Evaluation on the test split (E3 greedy, E4 beam) and 5. error analysis

In [ ]:
!python scripts/04_evaluate.py
!python scripts/05_error_analysis.py

In [ ]:
import pandas as pd
pd.read_csv('results/metrics/comparison.csv').round(3)

## 6. Download the trained models and results (to run the Streamlit app locally)

In [ ]:
!zip -q -r /content/eventgen_trained.zip models artifacts results data/splits data/processed/cleaning_report.json
from google.colab import files; files.download('/content/eventgen_trained.zip')